# humanoid-perception — 3D change detection between two room recordings

Compares **two phone videos of the same indoor room** (before / after) and reports
**added / removed / moved** objects in 3D, with open-vocabulary labels (SAM 3.1)
and a natural-language report.

**Pipeline:** reconstruct each clip (VGGT) → align with a Sim(3) using static
geometry → fuse into a visibility-aware voxel grid → diff → cluster → label.

**Runtime:** GPU (T4 is enough).  Runs end-to-end on a bundled *synthetic* room
with zero external data, or on your own clips.

> You need a Hugging Face token with access to the gated `facebook/sam3.1`
> checkpoint. Add it in Colab → *Secrets* (key icon) as `HF_TOKEN`.

In [ ]:
# 1) Check the runtime has a GPU
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 2) Install the project + heavy deps
REPO_URL = "https://github.com/noahleegithub/humanoid-perception.git"
WORKDIR = "/content/humanoid-perception"

import os
if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
%cd {WORKDIR}

!apt-get -qq install -y ffmpeg > /dev/null 2>&1
!pip -q install -e ".[dev]"
!pip -q install "transformers>=4.55" "huggingface_hub>=0.23"   # SAM 3.1
!pip -q install git+https://github.com/facebookresearch/vggt.git # VGGT
!pip -q install open3d                                           # global registration
print("install complete")

In [ ]:
# 3) Authenticate with Hugging Face (needed for the gated SAM 3.1 checkpoint)
import os
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        print("No HF_TOKEN found. Add it under Colab -> Secrets to enable SAM 3.1.", exc)
if token:
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("logged in to Hugging Face")
else:
    print("proceeding without SAM 3.1 (template labels only)")

## 4) Choose input data

* `synthetic` — generate a procedurally-edited room with **exact ground truth** (no data needed; great for a first run).
* `upload` — drag in two clips: `before.mp4` and `after.mp4`.
* `drive` — point at two clips already in your Google Drive.

In [ ]:
MODE = "synthetic"   # "synthetic" | "upload" | "drive"

VIDEO_A = VIDEO_B = DATASET = None

if MODE == "synthetic":
    from hp.synthetic import generate_dataset
    DATASET = "/content/data/raw/room01"
    generate_dataset(DATASET, n_frames=12, height=220, width=280, seed=0)
    print("synthetic dataset:", DATASET)

elif MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    names = sorted(uploaded)
    assert len(names) >= 2, "upload two clips (before + after)"
    VIDEO_A, VIDEO_B = names[0], names[1]
    print("videos:", VIDEO_A, VIDEO_B)

elif MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO_A = "/content/drive/MyDrive/room/before.mp4"
    VIDEO_B = "/content/drive/MyDrive/room/after.mp4"
    print(VIDEO_A, VIDEO_B)
else:
    raise ValueError(MODE)

## 5) Run the pipeline

Everything is config-driven (`configs/default.yaml`). Override any knob with
`--set key=value`-style entries. For real clips, reconstruction uses **VGGT** and
alignment uses **Open3D** global registration; for the synthetic dataset both are
bypassed (exact geometry, identity alignment).

In [ ]:
from hp.config import load_config
from hp.pipeline import run as run_pipeline

overrides = ["viz.backend=plotly"]
if MODE != "synthetic":
    # real clips: VGGT reconstruction, keep frames within the T4 memory budget
    overrides += ["recon.backend=vggt", "recon.frames_chunk_size=6", "io.max_keyframes=12"]
else:
    overrides += ["recon.backend=gt"]

cfg = load_config(overrides=overrides)
result = run_pipeline(
    video_a=VIDEO_A, video_b=VIDEO_B, dataset=DATASET,
    cfg=cfg, out_dir="/content/outputs/run1", align_method="auto",
)
cs = result.changeset
print("counts:", cs.counts())
print("outputs:", result.outputs)

In [ ]:
# 6) If running on the synthetic dataset, score against ground truth
if DATASET:
    import json
    from pathlib import Path
    from hp.eval.metrics import evaluate_changes
    gt = json.loads((Path(DATASET) / "gt.json").read_text())["changes"]
    ev = evaluate_changes(cs.clusters, gt, tol_m=0.6, allow_moved_as_two=True)
    print("evaluation vs ground truth:", ev)

## 7) Inspect the results

Interactive 3D view (unchanged = grey, **added = green**, **removed = red**, **moved = blue**),
then a table of changes and the generated Markdown report.

In [ ]:
import numpy as np
from hp.change import scene_world_points
from hp.viz.plotly_viz import change_figure

pts = np.concatenate([
    scene_world_points(result.scene_a, pixel_stride=4),
    scene_world_points(result.scene_b, pixel_stride=4),
], axis=0)
fig = change_figure(
    pts, cs, grid=result.grid_b, point_size=1.4, max_points=300000,
    colors=cfg.viz.get_path("colors"),
)
fig.update_layout(height=640)
fig.show()

In [ ]:
# 8) Table of changes + natural-language report
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

rows = []
for lc in cs.labeled:
    c = lc.cluster
    rows.append({
        "id": c.id,
        "kind": c.kind,
        "label": lc.label,
        "conf": round(float(lc.confidence), 2),
        "centre (m)": tuple(round(float(x), 2) for x in c.centroid),
        "size (cm)": tuple(int(round(float(x) * 100)) for x in c.size),
        "description": lc.description,
    })
display(pd.DataFrame(rows))

md_path = Path(result.outputs.get("markdown", ""))
if md_path.exists():
    display(Markdown(md_path.read_text()))

In [ ]:
# 9) Download the artefacts
from google.colab import files
for key in ("markdown", "json", "figure_html"):
    path = result.outputs.get(key)
    if path and Path(path).exists():
        files.download(path)

## 10) Using your own phone recordings

Follow [`docs/capture_protocol.md`](../docs/capture_protocol.md): a smooth ~30 s
walk around the room (translation, **not** standing in one spot rotating), both
passes covering every wall/object, fixed lighting, and 2–3 real changes
in between (move a chair, add a box, remove a mug).

Re-run cells **4 → 9** with `MODE` set to `upload` or `drive`.

**Tips**
* If alignment looks wrong (objects everywhere), the two clips had too little
  overlap — re-record with a fuller sweep of the room.
* A change that is *only* visible in one clip (occluded or never in frame in the
  other) is intentionally **not** reported.
* Tune resolution with `--set change.voxel_size=0.04` for finer detail (slower),
  or `0.08` for a smoother, more forgiving result.